# Math Prerequisites → Statistics | Chapter 0: Variance, Covariance & Correlation

> **Why this chapter exists:** The covariance matrix $\Sigma$ is the single object that PCA operates on. Every step of PCA — from the optimization problem to the interpretation of eigenvalues — traces back to understanding what covariance means. This chapter builds that understanding from the ground up: what variance tells us, what covariance captures, why the covariance matrix is the right summary of a multivariate dataset, and why we must standardize before running PCA.

---

## 1. Variance: Measuring Spread

### **Definition**
The **variance** of a random variable $X$ measures how much it spreads around its mean:

$$\text{Var}(X) = \sigma^2 = E[(X - \mu)^2] = E[X^2] - (E[X])^2$$

where $\mu = E[X]$ is the mean.

**For a finite sample** of $n$ observations $\{x_1, ..., x_n\}$, the **sample variance** (unbiased):
$$s^2 = \frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})^2$$

The $n-1$ denominator (Bessel's correction) corrects for the fact that we estimated the mean from the same data — dividing by $n$ would systematically underestimate the true variance.

**Standard deviation:** $\sigma = \sqrt{\text{Var}(X)}$ — same units as $X$, easier to interpret.

### **What Variance Tells You**
- **High variance:** Data points spread far from the mean. Lots of information in this direction.
- **Low variance:** Data points clustered near the mean. Little information in this direction.
- **Zero variance:** All data points identical. This feature/direction is completely uninformative.

### **PCA Connection**
PCA finds directions (principal components) that **maximize variance** of the projected data. The first PC captures the direction of highest variance — the direction with the most "spread", which contains the most information. This is why PCA works: it prioritizes informative dimensions.

---

## 2. Covariance: Measuring Joint Variation

### **Definition**
The **covariance** between two random variables $X$ and $Y$ measures how much they vary *together*:
$$\text{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)]$$

**Sample covariance:**
$$s_{XY} = \frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})$$

### **Interpreting the Sign and Magnitude**

| $\text{Cov}(X,Y)$ | Meaning |
| :---: | :--- |
| $> 0$ | When $X$ is above its mean, $Y$ tends to be above its mean too. Positive relationship. |
| $= 0$ | No linear relationship. $X$ and $Y$ vary independently (if jointly Gaussian, truly independent). |
| $< 0$ | When $X$ is above its mean, $Y$ tends to be below its mean. Negative relationship. |

**Covariance is symmetric:** $\text{Cov}(X, Y) = \text{Cov}(Y, X)$.

**Self-covariance = variance:** $\text{Cov}(X, X) = \text{Var}(X)$.

### **Problem: Covariance Depends on Scale**
If $X$ is measured in meters but $Y$ in kilometers: $\text{Cov}(X, Y)$ changes by factor 1000 if you convert $Y$ to meters. This makes raw covariance values hard to compare across different feature pairs.

---

## 3. Correlation: Scale-Free Covariance

### **Pearson Correlation**
$$\rho_{XY} = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y} \in [-1, +1]$$

Correlation is covariance normalized by the product of standard deviations — making it scale-invariant.

| $\rho$ | Interpretation |
| :---: | :--- |
| $+1$ | Perfect positive linear relationship |
| $+0.7$ | Strong positive linear relationship |
| $0$ | No linear relationship |
| $-0.7$ | Strong negative linear relationship |
| $-1$ | Perfect negative linear relationship |

### **Covariance vs Correlation in PCA**
PCA can be run on either the **covariance matrix** or the **correlation matrix**:

- **Covariance matrix PCA:** Directions of maximum absolute variance. Features measured in large units dominate (e.g., income in dollars dominates age in [0,100]).
- **Correlation matrix PCA:** Equivalent to standardizing first (zero mean, unit variance), then doing covariance PCA. All features contribute equally regardless of scale.

> **Rule:** Always use correlation matrix PCA (i.e., standardize first) unless your features are already in comparable units and variance differences are meaningful.

In sklearn: `StandardScaler()` → `PCA()` = correlation matrix PCA.

---

## 4. The Covariance Matrix: The Full Picture

For a dataset with $d$ features, the **covariance matrix** $\Sigma \in \mathbb{R}^{d \times d}$ collects all pairwise covariances:

$$\Sigma = \begin{bmatrix}
\sigma_1^2 & \text{Cov}(X_1, X_2) & \cdots & \text{Cov}(X_1, X_d) \\
\text{Cov}(X_2, X_1) & \sigma_2^2 & \cdots & \text{Cov}(X_2, X_d) \\
\vdots & & \ddots & \vdots \\
\text{Cov}(X_d, X_1) & \cdots & \cdots & \sigma_d^2
\end{bmatrix}$$

- **Diagonal entries:** $\Sigma_{ii} = \sigma_i^2$ = variance of feature $i$
- **Off-diagonal entries:** $\Sigma_{ij} = \text{Cov}(X_i, X_j)$ = covariance between features $i$ and $j$
- **Symmetric:** $\Sigma = \Sigma^T$ (since $\text{Cov}(X_i, X_j) = \text{Cov}(X_j, X_i)$)
- **Always PSD:** All eigenvalues $\geq 0$ (proved in Ch1 of linear algebra)

### **Sample Covariance Matrix Formula**
Given centered data $X \in \mathbb{R}^{N \times d}$ (each row = one data point, mean subtracted):
$$\hat{\Sigma} = \frac{1}{N-1} X^T X$$

This is a single compact formula that simultaneously computes all $d^2$ pairwise covariances.

### **What Off-Diagonal Structure Means**
- **All off-diagonal = 0:** Features are uncorrelated. PCA axes align with original feature axes — PCA does nothing useful.
- **Some large off-diagonal:** Features are correlated. PCA finds the rotation that diagonalizes $\Sigma$ — making them uncorrelated in PC space.
- **The bigger the off-diagonal entries, the more PCA can compress information** — because correlated directions can be combined.

### **Total Variance**
$$\text{Total variance} = \text{tr}(\Sigma) = \sum_{i=1}^d \sigma_i^2 = \sum_{i=1}^d \lambda_i$$

PCA redistributes total variance across PCs. It cannot create or destroy variance — only rearrange it into the most efficient order.

---

## 5. Why Standardize Before PCA?

Suppose your dataset has:
- Feature $A$: salary in USD, values in $[30{,}000, 200{,}000]$, variance $\approx 10^9$
- Feature $B$: age in years, values in $[20, 70]$, variance $\approx 200$

The covariance matrix:
$$\Sigma \approx \begin{bmatrix} 10^9 & \cdots \\ \cdots & 200 \end{bmatrix}$$

The first PC will be almost entirely salary — not because salary is more *informative*, but because it has a larger *scale*. PCA is dominated by unit choice, not information content.

**Standardization fix:** for each feature, subtract mean and divide by standard deviation:
$$z_i = \frac{x_i - \mu_i}{\sigma_i}$$

After standardization: all features have variance = 1. The covariance matrix of standardized data equals the **correlation matrix**:
$$\Sigma_{\text{std}} = \begin{bmatrix} 1 & \rho_{12} & \cdots \\ \rho_{21} & 1 & \cdots \\ \vdots & & \ddots \end{bmatrix}$$

Now PCA finds directions of maximum correlation-weighted variance — treating all features as equals.

> **Exception:** If features are in the same units and variance differences are meaningful (e.g., spectrometer readings at different wavelengths), use the raw covariance matrix.

---

## 6. Implementation Playground

In [ ]:
# ─── CELL 1: Covariance Matrix — Build, Visualize, Verify ─────────────────────
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
N = 500
# 4 features with known correlation structure
# Features: [height, weight, age, income]
mean = [170, 70, 35, 50000]
# True covariance structure
true_cov = np.array([
    [100,  60,   5,   500],   # height: correlated with weight
    [ 60, 225,  10,  2000],   # weight: moderate age corr, slight income corr
    [  5,  10, 100,  8000],   # age: correlated with income
    [500, 2000, 8000, 1e8]    # income: high variance, correlated with age
])
X = np.random.multivariate_normal(mean, true_cov, N)
features = ['Height', 'Weight', 'Age', 'Income']

# Sample covariance matrix
X_centered = X - X.mean(axis=0)
S = X_centered.T @ X_centered / (N - 1)  # same as np.cov(X.T)

print(f"Sample covariance matrix:")
print(np.array2string(S, formatter={'float_kind': lambda x: f'{x:10.1f}'}))
print(f"\nTrace (total variance): {np.trace(S):.1f}")
print(f"Verify: sum of variances = {sum(S[i,i] for i in range(4)):.1f}")

# Visualize covariance and correlation matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im1 = axes[0].imshow(np.log10(np.abs(S) + 1), cmap='Blues')
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(features, rotation=30)
axes[0].set_yticks(range(4)); axes[0].set_yticklabels(features)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{S[i,j]:.0f}', ha='center', va='center', fontsize=8,
                     color='white' if abs(S[i,j]) > 10000 else 'black')
axes[0].set_title('Covariance Matrix S\n(Income dominates due to scale)', fontsize=11)
plt.colorbar(im1, ax=axes[0])

# Correlation matrix
D_inv = np.diag(1 / np.sqrt(np.diag(S)))
R = D_inv @ S @ D_inv  # = correlation matrix

im2 = axes[1].imshow(R, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(features, rotation=30)
axes[1].set_yticks(range(4)); axes[1].set_yticklabels(features)
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f'{R[i,j]:.2f}', ha='center', va='center', fontsize=10)
axes[1].set_title('Correlation Matrix R\n(Scale-invariant, all diagonal = 1)', fontsize=11)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 2: Scale Effect on PCA — With vs Without Standardization ────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)
N = 300
# Two meaningful features: one in large units, one small
age    = np.random.normal(35, 10, N)       # range ~5-65, variance ~100
salary = 1000 * age + np.random.normal(0, 10000, N)  # correlated with age but huge variance

X = np.column_stack([age, salary])

# PCA without standardization
pca_raw = PCA(n_components=2).fit(X)

# PCA with standardization
X_std = StandardScaler().fit_transform(X)
pca_std = PCA(n_components=2).fit(X_std)

print("=== WITHOUT Standardization ===")
print(f"PC1 direction: {pca_raw.components_[0].round(5)}")
print(f"Explained variance ratio: {pca_raw.explained_variance_ratio_.round(4)}")
print(f"PC1 is almost entirely salary (large unit dominates)")

print("\n=== WITH Standardization ===")
print(f"PC1 direction: {pca_std.components_[0].round(4)}")
print(f"Explained variance ratio: {pca_std.explained_variance_ratio_.round(4)}")
print(f"PC1 roughly equal mix of age and salary (true correlation captured)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, pca, data, title in [
    (axes[0], pca_raw, X, 'Without Standardization'),
    (axes[1], pca_std, X_std, 'With Standardization')
]:
    ax.scatter(data[:, 0], data[:, 1], alpha=0.3, s=10, c='gray')
    mean = data.mean(axis=0)
    for i, (evec, eval_, color) in enumerate(zip(pca.components_, pca.explained_variance_, ['red', 'blue'])):
        scale = np.sqrt(eval_) * 2.5
        ax.annotate('', xy=mean + evec * scale, xytext=mean,
                    arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
        ax.text(*(mean + evec * scale), f' PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)',
                fontsize=9, color=color)
    ax.set_xlabel('Age' if i==0 else 'Age (scaled)'); ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()